# Train a vehicle detector (Car / Motorbike / Truck) on Colab

This notebook only **trains** and saves everything to your Google Drive. Evaluation on the held-out test split is done once, at the very end, by a separate script. **Never point this notebook at the test split.**

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Put `clean.zip` (the prepared dataset) and `code.zip` (this repo) in `MyDrive/vehicle-detection/`.

**Planned runs (fixed in advance, change only `RUN_NAME`, `MODEL`, `SEED` below):**

| RUN_NAME | MODEL | SEED | Purpose |
|---|---|---|---|
| `v8n_seed0` | `yolov8n.pt` | 0 | honest baseline |
| `v8n_seed1` | `yolov8n.pt` | 1 | measures seed-to-seed noise |
| `v8s_seed0` | `yolov8s.pt` | 0 | option A: bigger model |
| `v11n_seed0` | `yolo11n.pt` | 0 | option B: newer architecture, same size |
| spare | | | a crash, or a repeat of the winner with seed 1 |

If Colab disconnects, just run all cells again: unfinished runs resume from the last saved epoch, finished runs are skipped.

In [ ]:
# ---- EDIT ONLY THIS CELL ----
RUN_NAME = "v8n_seed0"
MODEL = "yolov8n.pt"  # pretrained COCO weights, downloaded automatically
SEED = 0
IMGSZ = 672
EPOCHS = 100
CODE_SOURCE = "drive_zip"  # "drive_zip" while the repo is private, "github" once it is public
GITHUB_REPO = "https://github.com/zohabaamir-ai/vehicle-detection-yolov8.git"
DRIVE_ROOT = "/content/drive/MyDrive/vehicle-detection"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip install -q ultralytics==8.4.174

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile

CODE_DIR = "/content/code"
if CODE_SOURCE == "drive_zip":
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    with zipfile.ZipFile(f"{DRIVE_ROOT}/code.zip") as z:
        z.extractall(CODE_DIR)
else:
    if not os.path.exists(CODE_DIR):
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, CODE_DIR], check=True)
for name in [m for m in sys.modules if m.startswith("vehicle_detection")]:
    del sys.modules[name]  # forget any older copy loaded earlier in this session
sys.path.insert(0, f"{CODE_DIR}/src")  # our package needs only numpy, pillow, pyyaml (in Colab)

import vehicle_detection

print("vehicle_detection", vehicle_detection.__version__, "loaded from", vehicle_detection.__file__)

In [ ]:
DATA_DIR = "/content/data/clean"  # keep this path: a resumed run remembers it
from vehicle_detection.verify import verify_dataset


def unpack_dataset():
    shutil.rmtree("/content/data", ignore_errors=True)
    os.makedirs("/content/data", exist_ok=True)
    with zipfile.ZipFile(f"{DRIVE_ROOT}/clean.zip") as z:
        z.extractall("/content/data")


def check_dataset():
    return verify_dataset(
        DATA_DIR, f"{CODE_DIR}/results/split_manifest.csv", f"{CODE_DIR}/results/split_summary.json"
    )


if not os.path.exists(DATA_DIR):
    unpack_dataset()
report = check_dataset()
if not report["ok"]:  # an older copy from earlier in this session: unpack the Drive copy again
    print("Dataset on this machine does not match; unpacking clean.zip again")
    unpack_dataset()
    report = check_dataset()
print(report)
assert report["ok"], "The dataset on Drive does not match the committed split manifest"

In [ ]:
from vehicle_detection.train import RunConfig, run_training

cfg = RunConfig(
    name=RUN_NAME,
    model=MODEL,
    data=f"{DATA_DIR}/data.yaml",
    project=f"{DRIVE_ROOT}/runs",
    seed=SEED,
    imgsz=IMGSZ,
    epochs=EPOCHS,
)
run_training(cfg)

In [ ]:
import json
import platform
from pathlib import Path

import torch
import ultralytics

from vehicle_detection.runs import read_results_csv, summarize_run
from vehicle_detection.train import run_dir

d = run_dir(cfg)
summary = summarize_run(read_results_csv(d / "results.csv"))
summary["run"] = {
    "name": cfg.name,
    "model": cfg.model,
    "seed": cfg.seed,
    "imgsz": cfg.imgsz,
    "epochs": cfg.epochs,
    "batch": cfg.batch,
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "python": platform.python_version(),
    "gpu": torch.cuda.get_device_name(0),
}
(d / "run_summary.json").write_text(json.dumps(summary, indent=2))

export = Path(DRIVE_ROOT) / "exports" / cfg.name
export.mkdir(parents=True, exist_ok=True)
for fname in ("results.csv", "args.yaml", "run_summary.json", "results.png"):
    if (d / fname).exists():
        shutil.copy(d / fname, export / fname)

print(json.dumps(summary, indent=2))
print("Small files copied to", export)

## Done with this run

Copy the printed summary back to the chat. The weights stay on Drive in `runs/<RUN_NAME>/weights/` (`best.pt` is the epoch with the best validation mAP@0.5:0.95). Then change `RUN_NAME`, `MODEL` and `SEED` in the first cell for the next planned run.